<a href="https://colab.research.google.com/github/popojinny/UROP/blob/main/2024%EC%9D%B4%EC%A0%84_%EC%A3%BC%EC%A0%9C_MCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q bertopic openpyxl sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.4 MB/s eta 0:00:00


In [9]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)

print("파일:", filename)
print("논문 수:", len(df))
print("열 이름:")
print(df.columns.tolist())

Saving 2024이전(포함).xlsx to 2024이전(포함).xlsx
파일: 2024이전(포함).xlsx
논문 수: 7069
열 이름:
['num', 'Unnamed: 1', '년도', '제목', '영문제목', 'Unnamed: 5', '초록', '영문초록', '서지정보', 'DOI']


In [10]:
filtered_df = df.dropna(subset=['num', '영문초록'])
docs = (
    filtered_df["영문초록"]
    .astype(str)
    .tolist()
)

print("초록 수:", len(docs))

초록 수: 5731


In [12]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=[
        "Mild Cognitive Impairment (MCI)"
    ],
    zeroshot_min_similarity=0.70,
    min_topic_size=25,
    verbose=True
)

topics, probabilities = topic_model.fit_transform(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-08-16 14:14:30,519 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/180 [00:00<?, ?it/s]

2026-08-16 14:27:41,984 - BERTopic - Embedding - Completed ✓
2026-08-16 14:27:41,986 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-16 14:27:52,901 - BERTopic - Dimensionality - Completed ✓
2026-08-16 14:27:52,902 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2026-08-16 14:27:52,949 - BERTopic - Zeroshot Step 1 - Completed ✓
2026-08-16 14:27:58,254 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-16 14:27:58,717 - BERTopic - Cluster - Completed ✓
2026-08-16 14:27:58,718 - BERTopic - Zeroshot Step 2 - Combining topics from zero-shot topic modeling with topics from clustering...
2026-08-16 14:27:58,750 - BERTopic - Zeroshot Step 2 - Completed ✓
2026-08-16 14:27:58,752 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-16 14:28:00,313 - BERTopic - Representation - Completed ✓


In [16]:
# BERTopic이 실제 분석한 문서에 해당하는 행만 선택
df_mci = df[df["영문초록"].notna()].copy()

# 길이 확인
print("BERTopic 분석 데이터:", len(df_mci))
print("BERTopic 결과:", len(topics))

BERTopic 분석 데이터: 5731
BERTopic 결과: 5731


In [17]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2034,-1_the_and_of_in,"[the, and, of, in, to, dementia, with, for, wa...",[Objectives: To identify the factors affecting...
1,0,7,Mild Cognitive Impairment (MCI),"[mci, stage, impairment, amnestic, are, alzhei...",[Aging causes deterioration of various aspects...
2,1,61,1_sleep_quality_and_rbd,"[sleep, quality, and, rbd, with, in, disorders...",[A study was conducted to confirm the changes ...
3,2,76,2_oral_health_dental_teeth,"[oral, health, dental, teeth, and, of, the, sw...",[Background: This study aimed to analyze corre...
4,3,46,3_delirium_fracture_hip_postoperative,"[delirium, fracture, hip, postoperative, patie...","[Purpose: To evaluate the incidence, risk fact..."
5,4,73,4_horticultural_therapy_program_the,"[horticultural, therapy, program, the, to, of,...",[This study purposed to examine the effect of ...
6,5,137,5_the_of_der_narrative,"[the, of, der, narrative, in, her, as, is, die...",[The purpose of this study is to examine the n...
7,6,105,6_smart_missing_to_the,"[smart, missing, to, the, is, system, and, inf...",[As the entry into an aging society progresses...
8,7,34,7_game_games_board_serious,"[game, games, board, serious, the, cognitive, ...","[As the elderly population increases, the numb..."
9,8,53,8_vr_virtual_reality_rehabilitation,"[vr, virtual, reality, rehabilitation, cogniti...",[This study attempted to present a leisure act...


In [18]:
df_mci["MCI_topic"] = [
    "YES" if topic == 0 else "NO"
    for topic in topics
]

print(df_mci["MCI_topic"].value_counts())

MCI_topic
NO     5724
YES       7
Name: count, dtype: int64


In [19]:
# BERTopic이 실제 분석한 초록이 있는 데이터만 사용
df_mci = df[df["영문초록"].notna()].copy()

# BERTopic 결과와 데이터 길이 확인
print("BERTopic 분석 데이터:", len(df_mci))
print("BERTopic 결과:", len(topics))

# Topic 0 = MCI
df_mci["MCI_topic"] = [
    "YES" if topic == 0 else "NO"
    for topic in topics
]

print("\nMCI 분류 결과")
print(df_mci["MCI_topic"].value_counts())

BERTopic 분석 데이터: 5731
BERTopic 결과: 5731

MCI 분류 결과
MCI_topic
NO     5724
YES       7
Name: count, dtype: int64


In [20]:
total_papers = len(df_mci)

mci_count = (df_mci["MCI_topic"] == "YES").sum()

mci_percentage = (
    mci_count / total_papers * 100
)

print("=" * 50)
print("MCI 분석 결과")
print("=" * 50)

print(f"BERTopic 분석 논문 수 : {total_papers:,}편")
print(f"MCI 주제 논문 수      : {mci_count:,}편")
print(f"MCI 주제 비율         : {mci_percentage:.2f}%")

MCI 분석 결과
BERTopic 분석 논문 수 : 5,731편
MCI 주제 논문 수      : 7편
MCI 주제 비율         : 0.12%


In [23]:
df_mci.columns = df_mci.columns.str.strip()

df_mci["년도"] = pd.to_numeric(
    df_mci["년도"],
    errors="coerce"
)

print(
    "연도 정보가 있는 논문:",
    df_mci["년도"].notna().sum()
)

print(
    "연도 정보가 없는 논문:",
    df_mci["년도"].isna().sum()
)

연도 정보가 있는 논문: 5731
연도 정보가 없는 논문: 0


In [25]:
year_df = df_mci.dropna(subset=["년도"]).copy()

year_df["년도"] = year_df["년도"].astype(int)

year_summary = (
    year_df
    .groupby("년도")
    .agg(
        전체논문수=("MCI_topic", "size"),
        MCI논문수=("MCI_topic", lambda x: (x == "YES").sum())
    )
    .reset_index()
)

year_summary["MCI비율"] = (
    year_summary["MCI논문수"]
    / year_summary["전체논문수"]
    * 100
)

year_summary

,년도,전체논문수,MCI논문수,MCI비율
0,2000,87,0,0.000000
1,2001,65,0,0.000000
2,2002,117,0,0.000000
3,2003,120,3,2.500000
4,2004,136,0,0.000000
5,2005,130,0,0.000000
6,2006,120,0,0.000000
7,2007,162,0,0.000000
8,2008,182,0,0.000000
9,2009,160,0,0.000000
